In [7]:
import json
from multiprocessing.resource_sharer import stop

import pandas as pd
from pathlib import Path

from transformation.station import Service

In [8]:
chemin = sorted(Path("../data/raw/").glob("prix-carburants_*.csv"))[-1]
print(chemin.name)

prix-carburants_20261010T183035Z.csv


In [9]:
df = pd.read_csv(chemin, dtype=str)

In [10]:
df.shape

(9814, 48)

In [11]:
print(df.columns)
CARBURANTS = ["Gazole", "SP95", "E85", "GPLc", "E10", "SP98"]

Index(['__id', 'id', 'latitude', 'longitude', 'Code postal', 'pop', 'Adresse',
       'Ville', 'horaires', 'services', 'prix', 'rupture', 'geom',
       'Prix Gazole mis à jour le', 'Prix Gazole', 'Prix SP95 mis à jour le',
       'Prix SP95', 'Prix E85 mis à jour le', 'Prix E85',
       'Prix GPLc mis à jour le', 'Prix GPLc', 'Prix E10 mis à jour le',
       'Prix E10', 'Prix SP98 mis à jour le', 'Prix SP98',
       'Début rupture e10 (si temporaire)', 'Type rupture e10',
       'Début rupture sp98 (si temporaire)', 'Type rupture sp98',
       'Début rupture sp95 (si temporaire)', 'Type rupture sp95',
       'Début rupture e85 (si temporaire)', 'Type rupture e85',
       'Début rupture GPLc (si temporaire)', 'Type rupture GPLc',
       'Début rupture gazole (si temporaire)', 'Type rupture gazole',
       'Carburants disponibles', 'Carburants indisponibles',
       'Carburants en rupture temporaire', 'Carburants en rupture definitive',
       'Automate 24-24 (oui/non)', 'Services propo

In [12]:
#verifie les identifiants sont unique et non nuls
print(df["id"].is_unique, df["id"].isna().sum())

True 0


In [13]:
#Derniere mis a jour des carburants
print("Fichier :", chemin.name)
for c in CARBURANTS:
    print(c, pd.to_datetime(df[f"Prix {c} mis à jour le"], utc=True).max())

Fichier : prix-carburants_20261010T183035Z.csv
Gazole 2026-10-10 14:59:17+00:00
SP95 2026-10-10 14:37:13+00:00
E85 2026-10-10 14:59:17+00:00
GPLc 2026-10-10 14:31:45+00:00
E10 2026-10-10 14:59:17+00:00
SP98 2026-10-10 14:59:17+00:00


In [14]:
lat = pd.to_numeric(df['latitude'], errors='coerce')
long = pd.to_numeric(df['longitude'], errors='coerce')

print(lat.describe())
print(long.describe())
print("manquantes :", lat.isna().sum(), long.isna().sum())

count    9.814000e+03
mean     4.668832e+06
std      2.253809e+05
min      4.139100e+06
25%      4.475780e+06
50%      4.696800e+06
75%      4.867575e+06
max      5.106500e+06
Name: latitude, dtype: float64
count      9814.000000
mean     268156.008583
std      284204.575623
min     -472300.000000
25%       66210.500000
50%      260850.000000
75%      490400.000000
max      954700.000000
Name: longitude, dtype: float64
manquantes : 0 0


In [15]:
for c in CARBURANTS:
    p = pd.to_numeric(df[f"Prix {c}"], errors='coerce')
    d = pd.to_datetime(df[f"Prix {c} mis à jour le"], utc=True)
    #print(f"prix {c} {p.isna().sum()} , date {d.isna().sum()}")
    print(f"min/max {p.min()}/{p.max()} | {(p<=0).sum()}")

min/max 2.09/2.916 | 0
min/max 1.549/2.74 | 0
min/max 0.69/2.259 | 0
min/max 0.825/2.31 | 0
min/max 1.88/2.689 | 0
min/max 1.98/2.88 | 0


In [16]:
print(df["pop"].value_counts(dropna=False))
print(sorted(df["code_departement"].dropna().unique()))
print(df["code_region"].value_counts(dropna=False))
print(df["Code postal"].str.len().value_counts(dropna=False))

pop
R    9376
A     438
Name: count, dtype: int64
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '21', '22', '23', '24', '25', '26', '27', '28', '29', '2A', '2B', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95']
code_region
84     1354
75     1141
76     1110
44      862
11      832
32      804
93      791
27      668
28      565
52      555
53      549
24      452
94      127
NaN       4
Name: count, dtype: int64
Code postal
5    9814
Name: count, dtype: int64


In [17]:
df['Département'].unique()

<StringArray>
[                  'Yonne',                   'Somme',
                  'Nièvre',                   'Isère',
                   'Doubs',                'Morbihan',
          'Hauts-de-Seine',            'Hautes-Alpes',
              'Val-d'Oise',       'Seine-Saint-Denis',
         'Alpes-Maritimes',                 'Aveyron',
                 'Gironde',                 'Hérault',
                  'Landes',            'Haute-Savoie',
                    'Oise',                     'Var',
                    'Nord',          'Seine-et-Marne',
 'Alpes-de-Haute-Provence',                 'Ardèche',
               'Côte-d'Or',                'Dordogne',
                 'Corrèze',                 'Essonne',
         'Tarn-et-Garonne',        'Bouches-du-Rhône',
                'Calvados',                  'Allier',
    'Pyrénées-Atlantiques',               'Finistère',
                'Yvelines',                  'Vienne',
                    'Cher',                    'Jur

In [18]:
df['Région'].unique()

<StringArray>
[   'Bourgogne-Franche-Comté',            'Hauts-de-France',
       'Auvergne-Rhône-Alpes',                   'Bretagne',
              'Île-de-France', 'Provence-Alpes-Côte d'Azur',
                  'Occitanie',         'Nouvelle-Aquitaine',
                  'Normandie',        'Centre-Val de Loire',
                  'Grand Est',                      'Corse',
           'Pays de la Loire',                          nan]
Length: 14, dtype: str

In [19]:
df['pop'].value_counts()

pop
R    9376
A     438
Name: count, dtype: int64

In [20]:
df['services']

0       {"service": ["Toilettes publiques", "Location ...
1       {"service": ["Boutique alimentaire", "Vente de...
2                                                     NaN
3                        {"service": "Automate CB 24/24"}
4       {"service": ["Toilettes publiques", "Laverie",...
                              ...                        
9809    {"service": ["Toilettes publiques", "Relais co...
9810                                                  NaN
9811    {"service": ["Laverie", "Relais colis", "Bouti...
9812    {"service": ["Boutique non alimentaire", "Stat...
9813    {"service": ["Toilettes publiques", "Station d...
Name: services, Length: 9814, dtype: str

In [21]:
print(df.loc[2, "Services proposés"])           
print(df["services"].dropna().iloc[0])                      # l'autre colonne, plus structurée ?

nan
{"service": ["Toilettes publiques", "Location de véhicule", "Vente de gaz domestique (Butane, Propane)", "Wifi", "DAB (Distributeur automatique de billets)"]}


In [22]:
print(df["services"].dropna().iloc[0])                      # l'autre colonne, plus structurée ?

{"service": ["Toilettes publiques", "Location de véhicule", "Vente de gaz domestique (Butane, Propane)", "Wifi", "DAB (Distributeur automatique de billets)"]}


In [23]:
df["geom"].iloc[0]

'48.183, 3.309'

In [24]:
for c in CARBURANTS:
    print(df[f"Prix {c}"].describe())

count     9139
unique     380
top       2.25
freq      1777
Name: Prix Gazole, dtype: object
count     2743
unique     252
top       1.99
freq       250
Name: Prix SP95, dtype: object
count      3711
unique      305
top       0.869
freq       1090
Name: Prix E85, dtype: object
count      1506
unique      232
top       0.999
freq        142
Name: Prix GPLc, dtype: object
count     7203
unique     323
top       1.99
freq      1255
Name: Prix E10, dtype: object
count     7168
unique     342
top       1.99
freq       908
Name: Prix SP98, dtype: object


In [25]:
from pathlib import Path
RACINE_PROJET = Path.cwd().parent

In [26]:
RACINE_PROJET

PosixPath('/home/etudiants/2026-2027/Projets/carburant-pipeline')

In [27]:
def construire_service(raw) ->Service:
    services = raw['services']

In [197]:
result = df[df['services'].notna()]
list_services = {}
d = json.loads(result.services[0])

for (current_id, elt) in zip(result.id, result.services):
    try:
        data = json.loads(elt)
        if current_id not in list_services:
            list_services[current_id] = []

        list_services[current_id]=data['service']

    except json.JSONDecodeError as e:
        # En Python, on lève l'exception d'origine ou un ValueError avec le message
        raise ValueError(e.msg)
len(list_services)

9141

In [147]:
df[df['services'].notna()]

{"service": ["Toilettes publiques", "Location de véhicule", "Vente de gaz domestique (Butane, Propane)", "Wifi", "DAB (Distributeur automatique de billets)"]}


In [141]:
df.services

'89100001'